### Data Prep

In [0]:
%sql
CREATE EXTERNAL TABLE IF NOT EXISTS kafka_poc.engagement.evolution_demo_external (
  `partition` INT,
  `offset` BIGINT,
  post_id STRING,
  `type` STRING,
  user_id STRING,
  ts BIGINT,
  device STRING,
  country STRING)
USING DELTA
LOCATION 's3://retail-data-lake-dev/delta_poc/data/evolution_demo/'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.enableRowTracking'     = 'true',
  'delta.parquet.compression.codec' = 'zstd')

In [0]:
%sql
MERGE INTO kafka_poc.engagement.evolution_demo_external AS t
USING (
  SELECT * FROM VALUES
    (0, 1001, 'p001', 'like',    'u01', 1733001000, 'ios',     'US'),
    (0, 1002, 'p002', 'view',    'u02', 1733001050, 'android', 'IN'),
    (1, 1003, 'p003', 'comment', 'u03', 1733001100, 'web',     'GB'),
    (1, 1004, 'p004', 'share',   'u04', 1733001150, 'ios',     'US'),
    (2, 1005, 'p005', 'like',    'u05', 1733001200, 'android', 'DE'),
    (2, 1006, 'p006', 'click',   'u06', 1733001250, 'web',     'IN'),
    (0, 1007, 'p007', 'view',    'u07', 1733001300, 'ios',     'FR'),
    (1, 1008, 'p008', 'like',    'u08', 1733001350, 'android', 'US'),
    (2, 1009, 'p009', 'comment', 'u09', 1733001400, 'web',     'IN'),
    (0, 1010, 'p010', 'share',   'u10', 1733001450, 'ios',     'GB')
  AS s(`partition`, `offset`, post_id, type, user_id, ts, device, country)
) AS s
ON t.post_id = s.post_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO kafka_poc.engagement.evolution_demo_external AS t
USING (
  SELECT * FROM VALUES
    -- 5 that will MATCH p001..p005 (changed type/offset/ts/device/country)
    (0, 2001, 'p001', 'share',   'u01', 1733005000, 'web',     'US'),
    (0, 2002, 'p002', 'like',    'u02', 1733005050, 'ios',     'IN'),
    (1, 2003, 'p003', 'view',    'u03', 1733005100, 'android', 'GB'),
    (1, 2004, 'p004', 'comment', 'u04', 1733005150, 'web',     'US'),
    (2, 2005, 'p005', 'click',   'u05', 1733005200, 'ios',     'DE'),
    -- 5 that will NOT MATCH p011..p015 (new)
    (2, 2011, 'p011', 'like',    'u11', 1733005250, 'android', 'IN'),
    (0, 2012, 'p012', 'view',    'u12', 1733005300, 'web',     'US'),
    (1, 2013, 'p013', 'comment', 'u13', 1733005350, 'ios',     'FR'),
    (2, 2014, 'p014', 'share',   'u14', 1733005400, 'android', 'GB'),
    (0, 2015, 'p015', 'click',   'u15', 1733005450, 'web',     'IN')
  AS s(`partition`, `offset`, post_id, type, user_id, ts, device, country)
) AS s
ON t.post_id = s.post_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
desc history kafka_poc.engagement.evolution_demo_external

In [0]:
%fs
ls s3://retail-data-lake-dev/delta_poc/data/evolution_demo/

### Physical Layout

In [0]:
%fs
ls s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_delta_log/

- If we see here, the version-1's JSON (..001.json) have all the files that has been added. So using this transaction log file it will point out to those added files.  

In [0]:
%fs
head s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_delta_log/00000000000000000001.json

In [0]:
%fs
head s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_delta_log/00000000000000000002.json

### Live files: 
If there are any changes to the file then new parquet will be created and Old parquet file will be marked as 'remove', 

And updated file will mark as 'add'.  

Now when we do 'select * version as of 2',  

Delta doesn't track "unchanged files" per version — it tracks adds and removes across the whole log, and a file is current if it was added earlier and never removed. The survivors come from commit 1, not commit 2.  

So here till version 2 we have all files, now Removed/modified in this version and added in this version(we get from ..002.json), so it will ignore those removed files but will read all rest of the files. 

In [0]:
%sql
SELECT DISTINCT _metadata.file_path
FROM delta.`s3://retail-data-lake-dev/delta_poc/data/evolution_demo`
--WHERE _metadata.file_path LIKE '%part-00000-6baecf93%';

In [0]:
%sql
select * from kafka_poc.engagement.evolution_demo_external

In [0]:
%fs
head s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_delta_log/00000000000000000002.crc

In [0]:
%fs
ls s3://retail-data-lake-dev/delta_poc/data/evolution_demo/_change_data/

### _staged_commits/
%md
_staged_commits/: is the staging area for catalog-managed commits — a newer Delta model (table feature catalogManaged, Delta 4.1+) where Unity Catalog, not the filesystem, decides what the latest committed version is. 

Table Properties: catalogManaged = True 

The lifecycle is stage → ratify → publish: 

Stage. When a writer commits, it first writes the commit's actions to _delta_log/_staged_commits/<version>.<uuid>.json — not straight to NN.json. 

Ratify. It then asks Unity Catalog to ratify that staged file. Once UC accepts it, that commit is official — the table is at that version even if the normal NN.json doesn't exist yet. UC is now the source of truth for "what's the latest version." 

Publish. Later, the staged file is published — literally copied from _staged_commits/<version>.<uuid>.json to _delta_log/<version>.json. 

Why do it this way. Recall the classic model from the notes: a writer commits by creating NN.json via put-if-absent, and the filesystem arbitrates the race (only one writer can create that exact name). Catalog-managed moves the arbitration to UC instead. That's the reason for the <uuid> in the staged filename — two writers racing for the same version don't collide on one filename; they each stage under a different uuid, and UC ratifies exactly one as the winner. The decision moves from "who won the filesystem PUT" to "who UC ratified." 

What happens to the two staged files. Both PUTs succeed (different uuids, no filename collision). Then both ask UC to ratify their file as version N. UC ratifies exactly one. The other is told "version N is already taken" — its staged file is now an orphan, never published, and that writer must retry: re-read the new latest version (N, the winner's), and attempt N+1. So "both commits will be true" is no — both staged, one committed. The loser does not silently land.. 

The working is still same only now UC will decide which file will survive. 

if the correctness guarantee is unchanged, why bother with UC at all? 

The scenario it solves: a write that must land in several tables, all-or-nothing. 

Example: 

> BEGIN TRANSACTION; 
> 
>   UPDATE accounts SET balance = balance - 100 WHERE id = 'A';   -- debit 
> 
>   UPDATE accounts SET balance = balance + 100 WHERE id = 'B';   -- credit 
> 
> COMMIT; 

 

Why a filesystem PUT can't do this: each UPDATE is a separate commit to a separate table's _delta_log/. Two independent put-if-absent operations on two different logs have no shared outcome — one can succeed and the other fail. 

There's no single atomic step spanning both. UC provides that step: it ratifies the staged commits for both tables as one unit, or rolls both back. 

More realistic data-engineering versions of the same shape: 

Upsert + audit together. MERGE into your fact table, and in the same transaction INSERT a row into an audit/ledger table. You never end up with data changed but unlogged, or logged but not changed. 

Fact + rollup. Append to a detail table and update a pre-aggregated summary table atomically, so the summary is never briefly out of sync with the detail. 

Move between zones. Delete from a staging table and insert into the curated table as one unit — no window where the row exists in both or neither. 

 

In [0]:
%sql
SHOW TBLPROPERTIES kafka_poc.engagement.evolution_demo_external;
-- look for delta.feature.catalogManaged = supported

### _change_data/

In [0]:
%fs
ls 